# Closing figures for the TFM — connectome structure ↔ function via ESNs

**Author:** Alejandro de Haro García · **Advisor:** M. Á. Serrano (UB)

This single notebook reproduces the **two closing figures** of the thesis and the
secondary analyses that support them, starting from `data_with_metrics.pkl` and the
parameterised C++ ESN engines in `./cpp/`.

**Design principles (so it runs autonomously on Windows and is cheap to re-run):**

1. **One parameter cell.** Every knob — `TAU`, `RIDGE`, `TRAIN_RATIO`, `STEPS`,
   reservoir size, replicate counts, `AGE_SPLIT`, paths, thread count — lives in
   **Section 0**. The C++ reads the same values from a generated `config.txt`, so
   nothing is hard-coded and nothing needs recompiling when a parameter changes.
2. **Heavy compute is isolated and cached.** The C++ runs in dedicated cells whose
   results are written to `.pkl`; on a re-run those cells *load the cache* instead of
   recomputing, so you can edit the figures freely without paying the compute again.
   Delete the cache file (or set `FORCE_RECOMPUTE = True`) to recompute.
3. **Primary metric = Memory Capacity, `MC = Σ_τ r²(τ)` with `τ = 20`** (Damicelli's
   r², *not* Suárez's r). The r-variant is computed only as a consistency check.
4. **Plot conventions match `firstpart.ipynb` exactly** (paper rcParams, log/geometric
   bins, `groupby('age').mean()` lifespan trajectories — never median bins).

**Figure 3 (primary):** *ESN performance and age as a function of disparity* —
(a) the lifespan coupling of MC and disparity, (b) MC against weight surrogates,
(c) disparity / clustering / communicability as the structural drivers of MC.

**Figure 4 (primary):** mediation placeholder + development-vs-aging correlation
structure + PCA + UMAP (incl. a UMAP coloured by dataset as a sanity check).

> Run the cells in order. The first full run compiles the C++ and executes the MC
> sweep (the only slow step); everything afterwards is fast and re-runs from cache.


## 0 · Parameters — the single source of truth

Edit here. These values are echoed into `cpp/config.txt` for the C++ engines, so the
Python analysis and the compiled engines always agree.

In [ ]:
# ============================ EDITABLE PARAMETERS ============================
from pathlib import Path

# ---- ESN / Memory-Capacity (PRIMARY metric) --------------------------------
RESERVOIR_SIZE  = 90          # nodes in the connectome = reservoir size
SPECTRAL_RADIUS = 0.99        # ρ(W) after rescaling (Suárez convention)
TAU             = 20          # MC delays summed: MC = Σ_{τ=1..TAU} r²(τ)   ← PRIMARY = 20
RIDGE           = 1e-4        # ridge (Tikhonov) regularisation of the readout
TRAIN_RATIO     = 0.7         # train/test split of the driven states
STEPS           = 6000        # length of the input sequence u(t) ~ U(-1,1)
WASHOUT         = 100         # transient discarded before fitting
WIN_REPS        = 5           # random input-projection (W_in) replicates, averaged
NULL_REPS       = 10          # surrogate replicates per subject (stochastic nulls)
SEED            = 42          # base RNG seed (per-ESN seeds derive from this + sid)

# ---- Surrogate / null models (Section 5b, 7a) ------------------------------
DO_R            = True        # also compute the Σ|r| variant (r-vs-r² check, 7e)
DO_BS           = True        # broken-stick  = disparity null model
DO_UNI          = True        # row-uniform   = "H0" (no weight heterogeneity)
DO_RSH          = True        # reshuffle     = "H1" (wiring vs weights)
RESHUFFLE_MODE  = "node"      # "node" (paper H1, per-node) | "global"

# ---- Biological (thalamic) input & random-pair null ------------------------
THAL_NODES      = [76, 77]    # input projected to thalamic nodes (AAL90 L/R thalamus)
N_RAND_PAIRS    = 50          # random input-pair null for the bio MC (0 = off)

# ---- Information Processing Capacity (Section 7c) ---------------------------
IPC_TAU         = 6           # SMALL: cross terms are O(tau²)
IPC_CROSS21     = False       # include P2·P1 cross terms (O(tau²), optional)
IPC_CROSS22     = False       # include P2·P2 cross terms (optional)

# ---- Analysis ---------------------------------------------------------------
AGE_SPLIT       = 33          # development (≤) vs aging (>). See FLAG in §3.
FEATURES        = ['Ratio', 'MC_Glob', 'comm_mean', 'C_w', 'age']   # PCA/UMAP feature set
SUBSET_N        = None        # None = all subjects; an int = random subset (smoke test)
DO_FULL_HORSERACE = False     # §7f: 16-metric horse-race (networkx, slow). Off by default.

# ---- Mijalkov et al. replication (Section 7d) ------------------------------
MIJALKOV_DATASET = 8          # camCAN
MIJALKOV_DENSITY = 0.15       # proportional threshold density

# ---- Infrastructure --------------------------------------------------------
MSYS2_ROOT      = r"C:\msys64"            # MSYS2 install (provides mingw64 g++ + Eigen)
N_WORKERS       = 0                        # 0 = use all cores for the C++ (OpenMP)
FORCE_RECOMPUTE = False                    # True = ignore caches and recompute everything

DATA_PKL  = Path("../procdata/data_with_metrics.pkl")

CACHE_DIR = Path("cache");   CACHE_DIR.mkdir(exist_ok=True)
FIG_DIR   = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

MC_CACHE       = CACHE_DIR / "mc_results.pkl"
IPC_CACHE      = CACHE_DIR / "ipc_results.pkl"
MIJALKOV_CACHE = CACHE_DIR / "mijalkov_mc.pkl"
HORSERACE_CACHE= CACHE_DIR / "horserace_metrics.pkl"

# Dataset id → short name (matches firstpart.ipynb)
DS_NAMES = {1:'dHCP', 2:'BCP', 3:'CALM', 4:'RED', 5:'ACE',
            6:'HCPd', 7:'HCPya', 8:'camCAN', 9:'HCPa'}

print("Parameters loaded.")
print(f"  PRIMARY metric: MC = Σ r²(τ), τ=1..{TAU} | ridge={RIDGE} | train={TRAIN_RATIO} | steps={STEPS}")
print(f"  reservoir={RESERVOIR_SIZE} | ρ={SPECTRAL_RADIUS} | win_reps={WIN_REPS} | null_reps={NULL_REPS}")
print(f"  AGE_SPLIT={AGE_SPLIT} | SUBSET_N={SUBSET_N} | thal_nodes={THAL_NODES} | n_rand_pairs={N_RAND_PAIRS}")


Parameters loaded.
  PRIMARY metric: MC = Σ r²(τ), τ=1..20 | ridge=0.0001 | train=0.7 | steps=6000
  reservoir=90 | ρ=0.99 | win_reps=5 | null_reps=10
  AGE_SPLIT=33 | SUBSET_N=None | thal_nodes=[76, 77] | n_rand_pairs=50


## 1 · Imports & plot style

The rcParams and theme are copied verbatim from `firstpart.ipynb` so every figure in
the thesis shares one visual identity.

In [9]:
import os, sys, time, json, platform, subprocess, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from scipy.linalg import expm, eigvals

# UMAP is optional; the UMAP panels degrade gracefully if it is not installed.
try:
    import umap
    HAVE_UMAP = True
except Exception:
    HAVE_UMAP = False
    print("note: umap-learn not found — UMAP panels will be skipped. `pip install umap-learn` to enable.")

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# networkx only needed for the optional 16-metric supplementary horse-race (§7f)
try:
    import networkx as nx
    HAVE_NX = True
except Exception:
    HAVE_NX = False

# ── EXACT plot conventions from firstpart.ipynb ──────────────────────────────
sns.set_theme(style='ticks', context='paper')
plt.rcParams.update({
    'font.size':7, 'axes.titlesize':7, 'axes.labelsize':7,
    'xtick.labelsize':6, 'ytick.labelsize':6,
    'lines.linewidth':0.9, 'lines.markersize':2.5, 'figure.dpi':150,
})

def make_ds_colors(ds_list):
    """{dataset_id: color} on tab10 — identical to firstpart.ipynb."""
    cmap = plt.cm.tab10
    return {ds: cmap(i / max(len(ds_list) - 1, 1)) for i, ds in enumerate(ds_list)}

print("Imports OK. UMAP:", HAVE_UMAP, "| networkx:", HAVE_NX)


Imports OK. UMAP: True | networkx: True


## 2 · Helpers

Re-implements the pieces `helperfuncs.py` provided (it was not part of the export),
plus the structural metrics, kept consistent with `firstpart.ipynb`.

In [7]:
# ── Nodal metrics (verbatim from firstpart.ipynb) ────────────────────────────
def node_metrics(adj):
    """Per-node metrics of a weighted adjacency matrix.
    Returns k (degree), s (strength), Y (disparity), kY (k·Y), w (mean weight)."""
    A = np.asarray(adj, float).copy(); np.fill_diagonal(A, 0)
    k  = (A > 0).sum(axis=1).astype(float)
    s  = A.sum(axis=1)
    p  = A / np.where(s > 0, s, 1.0)[:, None]
    Yi = (p**2).sum(axis=1)
    wi = np.divide(s, k, out=np.zeros_like(s, dtype=float), where=k > 0)
    return k, s, Yi, k * Yi, wi

# ── Weighted clustering (Onnela 2005) & communicability — as in firstpart ────
def onnela_clustering_global(W):
    W = np.asarray(W, float)
    w_max = W.max()
    if w_max == 0: return 0.0
    W_cbrt = np.cbrt(W / w_max)
    k = (W > 0).sum(axis=1).astype(float)
    num = np.sum((W_cbrt @ W_cbrt) * W_cbrt, axis=1)
    denom = k * (k - 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        c_i = np.where(denom > 0, num / denom, 0.0)
    return float(c_i.mean())

def communicability_mean(A, rho=SPECTRAL_RADIUS):
    A = np.asarray(A, float).copy(); np.fill_diagonal(A, 0)
    N = A.shape[0]
    sr = np.abs(eigvals(A)).max()
    A_norm = A * (rho / sr) if sr > 0 else A
    C = expm(A_norm)
    return float((C.sum() - np.trace(C)) / (N * (N - 1)))

# ── Linear fit: R² and overall-model p (replaces helperfuncs.fit_linear) ─────
def fit_linear(y, X):
    """OLS with intercept. Returns (R², p_F). X is (n,) or (n,p)."""
    y = np.asarray(y, float); X = np.asarray(X, float)
    if X.ndim == 1: X = X[:, None]
    A = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    yhat = A @ beta
    ss_res = float(((y - yhat) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    n, p = len(y), X.shape[1]
    if n - p - 1 > 0 and 0 < r2 < 1:
        F = (r2 / p) / ((1 - r2) / (n - p - 1)); pval = float(stats.f.sf(F, p, n - p - 1))
    else:
        pval = np.nan
    return r2, pval

# ── Partial correlation (verbatim from ESN.ipynb) ────────────────────────────
def partial_r(df_, x, y, cov):
    sub = df_[[x, y, cov]].dropna()
    def resid(a, b):
        c = np.polyfit(sub[b], sub[a], 1)
        return sub[a].values - np.polyval(c, sub[b].values)
    return pearsonr(resid(x, cov), resid(y, cov))

# ── Lifespan trajectory: groupby('age').mean() + bootstrap CI (NO median bins) ─
def age_trajectory(df, col, n_boot=500, min_n=3, seed=42):
    """Mean of `col` per integer age (firstpart convention) with a bootstrap 95% CI
    band of the mean. Returns x (ages), m (means), lo, hi."""
    sub = df.dropna(subset=[col, 'age'])
    g = sub.groupby('age')[col]
    x = np.array(sorted(g.groups.keys()), float)
    m = g.mean().reindex(x).values
    rng = np.random.default_rng(seed)
    lo = np.empty_like(m); hi = np.empty_like(m)
    for i, a in enumerate(x):
        v = sub.loc[sub['age'] == a, col].values
        if len(v) < min_n:
            lo[i] = hi[i] = m[i]
        else:
            bs = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
            lo[i], hi[i] = np.percentile(bs, [2.5, 97.5])
    return x, m, lo, hi

def safe_div(a, b):
    return a / b if (b is not None and b != 0) else np.nan

# ── Proportional threshold (Mijalkov replication, §7d) ───────────────────────
def proportional_threshold(A, density=0.15):
    """Keep the strongest `density` fraction of the possible undirected edges."""
    A = np.asarray(A, float).copy(); np.fill_diagonal(A, 0)
    n = A.shape[0]; iu = np.triu_indices(n, 1)
    w = A[iu]
    keep = int(round(density * len(w)))
    if keep <= 0 or (w > 0).sum() == 0:
        return np.zeros_like(A)
    thr = np.sort(w)[::-1][min(keep, len(w)) - 1]
    M = np.where(A >= thr, A, 0.0)
    M = np.triu(M, 1); M = M + M.T
    return M

print("Helpers ready.")


Helpers ready.


## 3 · Load the data

Loads `data_with_metrics.pkl` (a `DataFrame` with one row per subject) and attaches a
stable integer `sid` (= row order) used to merge the C++ outputs back. The provided
table has no subject-id column, so `sid` is the canonical key on both sides.

> **FLAG — `AGE_SPLIT`.** Both notebooks use **33**; the manuscript text/Table III
> describe a development/aging turning point near **30–32**. We default to 33 to match
> the existing analyses and expose it as a parameter. Change `AGE_SPLIT` in §0 to test
> sensitivity.

In [10]:
# Robust load of the pickled DataFrame.
def load_dataframe(path):
    try:
        return pd.read_pickle(path)
    except Exception:
        with open(path, 'rb') as f:
            return pickle.load(f)

assert DATA_PKL.exists(), f"Data not found: {DATA_PKL.resolve()}"
df = load_dataframe(DATA_PKL)
df = df.reset_index(drop=True)
df['sid'] = np.arange(len(df), dtype=int)          # stable merge key

expected = {'age','dataset','sex','connectome','decade','density','strength','degree',
            'Y_obs','kY_obs','Y_null','Ratio','C_unw','C_w','comm_mean'}
missing = expected - set(df.columns)
if missing:
    print("WARNING: expected columns missing from the pickle:", sorted(missing))

# Optional smoke-test subset (keeps the full pipeline path, just fewer subjects).
if SUBSET_N is not None and SUBSET_N < len(df):
    df = df.sample(n=SUBSET_N, random_state=SEED).sort_values('sid').reset_index(drop=True)
    print(f"SUBSET_N active: using {len(df)} subjects.")

# Convenience: per-subject development/aging label and dataset colours.
df['stage'] = np.where(df['age'] <= AGE_SPLIT, 'Development', 'Aging')
ds_list = sorted(df['dataset'].unique())
ds2col  = make_ds_colors(ds_list)

print(f"Loaded {len(df)} subjects | datasets: {[DS_NAMES.get(d, d) for d in ds_list]}")
print(f"age range [{df.age.min():.0f}, {df.age.max():.0f}] | "
      f"Development n={(df.stage=='Development').sum()}, Aging n={(df.stage=='Aging').sum()}")
df[['sid','age','dataset','sex','Ratio','kY_obs','C_w','comm_mean']].head()


Loaded 3901 subjects | datasets: ['dHCP', 'BCP', 'CALM', 'RED', 'ACE', 'HCPd', 'HCPya', 'camCAN', 'HCPa']
age range [0, 90] | Development n=2546, Aging n=1355


,sid,age,dataset,sex,Ratio,kY_obs,C_w,comm_mean
0,0,0,1,0,1.755841,3.307174,0.016887,0.011064
1,1,0,1,0,1.947808,3.698450,0.015413,0.010491
2,2,0,1,0,1.870164,3.584762,0.016119,0.011139
3,3,0,1,1,1.778583,3.354811,0.017793,0.010728
4,4,0,1,0,1.848612,3.511614,0.012787,0.010231


## 4 · C++ ESN pipeline (isolated & cached)

The heavy compute is delegated to the two compiled engines in `./cpp/`. The driver
below (a) compiles them on demand, (b) writes `config.txt` from the §0 parameters,
(c) exports the connectomes as `cpp/data/connectomes.csv` keyed by `sid`, and
(d) runs the engine and merges the result back into `df`.

Each *run* cell caches its merged result to `cache/*.pkl` and **loads the cache on a
re-run** (unless `FORCE_RECOMPUTE = True`). The MC sweep (§4b) is the only slow step.

In [19]:
# ── Platform / toolchain helpers ─────────────────────────────────────────────
def _is_windows():
    return platform.system().lower().startswith("win")


def _gpp_and_eigen():
    gpp   = r"C:\msys64\ucrt64\bin\g++.exe"
    eigen = r"C:\rtools45\x86_64-w64-mingw32.static.posix\include\eigen3"
    binp  = r"C:\msys64\ucrt64\bin"
    return gpp, eigen, binp

def _exe_name(stem):
    return stem + (".exe" if _is_windows() else "")

def _needs_build(exe, sources):
    exe = Path(exe)
    if not exe.exists(): return True
    t = exe.stat().st_mtime
    return any(Path(s).stat().st_mtime > t for s in sources)

COMMON_SRC = ["ESN.cpp", "Math.cpp", "Utils.cpp"]
COMMON_HDR = ["ESN.h", "Task.h", "Config.h"]

def compile_cpp(stems=("esn_run", "esn_ipc")):
    """Compile the requested engine stems in CPP_DIR. Skips up-to-date binaries.
    Returns {stem: Path(exe)}."""
    gpp, eigen, binp = _gpp_and_eigen()
    if not Path(gpp).exists() and _is_windows():
        raise RuntimeError(f"g++ not found at {gpp}. Install MSYS2 + mingw-w64-x86_64-gcc, "
                           f"or fix MSYS2_ROOT in §0.")
    if not eigen or not Path(eigen).exists():
        raise RuntimeError("Eigen headers not found. On Windows: `pacman -S "
                           "mingw-w64-x86_64-eigen3`; elsewhere set EIGEN_INCLUDE.")
    flags = ["-std=c++20", "-O3", "-fopenmp", f"-I{eigen}",
             "-Wno-deprecated-declarations", "-Wno-deprecated-enum-enum-conversion"]
    env = dict(os.environ)
    if binp: env["PATH"] = binp + os.pathsep + env.get("PATH", "")
    srcs = {"esn_run": "esn_run.cpp", "esn_ipc": "esn_ipc.cpp"}
    out = {}
    for stem in stems:
        exe =  _exe_name(stem)
        dep = [ s for s in COMMON_SRC + COMMON_HDR + [srcs[stem]]]
        if _needs_build(exe, dep):
            cmd = [gpp, *flags, srcs[stem], *COMMON_SRC, "-o", _exe_name(stem)]
            print(f"compiling {stem} …")
            r = subprocess.run(cmd, env=env,
                               capture_output=True, text=True)
            if r.returncode != 0:
                print(r.stdout); print(r.stderr)
                raise RuntimeError(f"compilation of {stem} failed")
        else:
            print(f"{stem}: up to date")
        out[stem] = exe
    return out

# ── config.txt writer (merges §0 parameters with per-run overrides) ──────────
def write_config(**overrides):
    cfg = dict(
        reservoir_size=RESERVOIR_SIZE, spectral_radius=SPECTRAL_RADIUS, tau=TAU,
        ridge=RIDGE, train_ratio=TRAIN_RATIO, steps=STEPS, washout=WASHOUT,
        win_reps=WIN_REPS, null_reps=NULL_REPS, n_rand_pairs=N_RAND_PAIRS, seed=SEED,
        do_r=str(DO_R).lower(), do_bs=str(DO_BS).lower(), do_uni=str(DO_UNI).lower(),
        do_rsh=str(DO_RSH).lower(), reshuffle_mode=RESHUFFLE_MODE,
        thal_nodes=",".join(map(str, THAL_NODES)),
        ipc_tau=IPC_TAU, ipc_cross21=str(IPC_CROSS21).lower(),
        ipc_cross22=str(IPC_CROSS22).lower(),
        in_csv="data/connectomes.csv", out_csv="mc_results.csv",
        ipc_out_csv="ipc_results.csv",
        threads=(N_WORKERS if N_WORKERS and N_WORKERS > 0 else os.cpu_count() or 4),
    )
    cfg.update(overrides)
    path =  "config.txt"
    with open(path, "w") as f:
        f.write("# auto-generated from the notebook's Section 0\n")
        for k, v in cfg.items():
            f.write(f"{k} = {v}\n")
    return path

# ── connectome exporter (sid + flattened N×N, row-major; header) ─────────────
def export_connectomes(sub_df, fname="connectomes.csv", n=RESERVOIR_SIZE):
    arrs = np.stack([np.asarray(c, float) for c in sub_df['connectome'].values])
    assert arrs.shape[1:] == (n, n), f"connectome shape {arrs.shape[1:]} != ({n},{n})"
    flat = arrs.reshape(arrs.shape[0], -1)                      # row-major
    M = np.column_stack([sub_df['sid'].values.astype(int), flat])
    header = "subject_id," + ",".join(f"w{i}" for i in range(n * n))
    ( "data").mkdir(exist_ok=True)
    path =  "data" / fname
    np.savetxt(path, M, delimiter=",", header=header, comments="", fmt="%.6g")
    return path

def run_exe(exe, config="config.txt"):
    gpp, eigen, binp = _gpp_and_eigen()
    env = dict(os.environ)
    if binp: env["PATH"] = binp + os.pathsep + env.get("PATH", "")
    t0 = time.time()
    proc = subprocess.Popen([str(Path(exe).resolve()), config],
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{exe} exited with code {proc.returncode}")
    print(f"\n[{Path(exe).name}] finished in {time.time()-t0:.1f}s")

print("C++ driver ready.")


C++ driver ready.


### 4b · Memory-Capacity sweep — the slow step (cached)

Computes, in one pass per subject: real `MC_Glob` / `MC_Bio`, the three weight
surrogates × {global, bio} with their MC ratios, the optional Σ|r| variant, and the
thalamic random-pair null (`MC_Rand_*`, `Z_Score`). Result is merged on `sid` and
cached.

> **Scaling tip.** Surrogates × `NULL_REPS` dominate the cost. For a first pass set
> `DO_BS = DO_UNI = DO_RSH = False` (and `N_RAND_PAIRS = 0`) to get the real MC quickly;
> enable them later for the null panels — the cache makes the second run incremental.

In [ ]:
if MC_CACHE.exists() and not FORCE_RECOMPUTE:
    df = load_dataframe(MC_CACHE)
    print(f"Loaded MC cache ({len(df)} subjects) → {MC_CACHE}")
else:
    compile_cpp(["esn_run"])
    export_connectomes(df, "connectomes.csv")
    write_config(out_csv="mc_results.csv")
    run_exe( ("esn_run.exe" if _is_windows() else "esn_run"))
    mc = pd.read_csv( "mc_results.csv").rename(columns={"subject_id": "sid"})
    # Drop any pre-existing MC columns before merging (idempotent re-runs).
    df = df.drop(columns=[c for c in mc.columns if c != "sid" and c in df.columns],
                 errors="ignore")
    df = df.merge(mc, on="sid", how="left")
    df.to_pickle(MC_CACHE)
    print(f"Saved MC cache → {MC_CACHE}")

mc_cols = [c for c in df.columns if c.startswith(('MC_', 'Ratio_', 'Z_Score'))]
print("MC columns available:", mc_cols)
df[['sid','age','MC_Glob','MC_Bio']].describe().loc[['mean','std','min','max']]


compiling esn_run …


In [13]:
import shutil, os, glob
from pathlib import Path
print("g++ on PATH :", shutil.which("g++"))
print("gcc on PATH :", shutil.which("gcc"))
print("cl  on PATH :", shutil.which("cl"))      # MSVC / Visual Studio
for root in [r"C:\msys64", r"C:\msys2", r"C:\mingw64", r"C:\w64devkit", r"C:\TDM-GCC-64",
             os.path.expanduser(r"~\scoop\apps\msys2\current")]:
    for sub in ("mingw64", "ucrt64", ""):
        p = Path(root) / sub / "bin" / "g++.exe"
        if p.exists(): print("found g++ :", p)
for hit in glob.glob(r"C:\**\eigen3\Eigen\Dense", recursive=True)[:5]:
    print("found Eigen:", Path(hit).parents[1])

g++ on PATH : None
gcc on PATH : None
cl  on PATH : None
found g++ : C:\msys64\ucrt64\bin\g++.exe
found Eigen: C:\rtools45\usr\lib\mxe\usr\x86_64-w64-mingw32.static.posix\include\eigen3
found Eigen: C:\rtools45\x86_64-w64-mingw32.static.posix\include\eigen3


In [ ]:
import shutil
EIGEN_INCLUDE = ""   # optional: set to the folder that CONTAINS the 'Eigen' directory, if auto-detect fails

def _find_gpp():
    cands = [Path(MSYS2_ROOT) / sub / "bin" / ("g++.exe" if _is_windows() else "g++")
             for sub in ("mingw64", "ucrt64")]
    for p in cands:
        if p.exists(): return str(p)
    w = shutil.which("g++")                       # anything already on PATH
    if w: return w
    if _is_windows():
        for root in (r"C:\msys64", r"C:\msys2", r"C:\mingw64", r"C:\w64devkit", r"C:\TDM-GCC-64"):
            for sub in ("mingw64", "ucrt64", ""):
                p = Path(root) / sub / "bin" / "g++.exe"
                if p.exists(): return str(p)
    return ""

def _find_eigen(gpp_path):
    cands = [EIGEN_INCLUDE, os.environ.get("EIGEN_INCLUDE", "")]
    if gpp_path:
        cands.append(str(Path(gpp_path).parents[1] / "include" / "eigen3"))
    cands += [r"C:\msys64\mingw64\include\eigen3", r"C:\msys64\ucrt64\include\eigen3",
              "/usr/include/eigen3", "/usr/local/include/eigen3", "/opt/homebrew/include/eigen3"]
    for c in cands:
        if c and (Path(c) / "Eigen" / "Dense").exists(): return c
    return ""

def _gpp_and_eigen():
    gpp = _find_gpp()
    return gpp, _find_eigen(gpp), (str(Path(gpp).parent) if gpp else "")


### 4c · Information Processing Capacity (cached)

Legendre P1/P2 + cross terms (as r²) at a small `IPC_TAU`. Supports the secondary
claim that MC (the linear P1 term) captures the bulk of the capacity and that the
order decomposition is consistent across the lifespan (§7c).

In [ ]:
if IPC_CACHE.exists() and not FORCE_RECOMPUTE:
    df_ipc = load_dataframe(IPC_CACHE)
    print(f"Loaded IPC cache ({len(df_ipc)} subjects) → {IPC_CACHE}")
else:
    compile_cpp(["esn_ipc"])
    # connectomes.csv already written by §4b for the same df; re-export if absent.
    if not ( "data" / "connectomes.csv").exists():
        export_connectomes(df, "connectomes.csv")
    write_config(ipc_out_csv="ipc_results.csv")
    run_exe( ("esn_ipc.exe" if _is_windows() else "esn_ipc"))
    ipc = pd.read_csv( "ipc_results.csv").rename(columns={"subject_id": "sid"})
    df_ipc = df[['sid','age','stage','dataset']].merge(ipc, on="sid", how="left")
    df_ipc.to_pickle(IPC_CACHE)
    print(f"Saved IPC cache → {IPC_CACHE}")

print("IPC columns:", [c for c in df_ipc.columns if c not in ('sid','age','stage','dataset')])
df_ipc.head()


### 4d · Mijalkov replication input (cached)

Runs the MC engine on **proportionally thresholded** camCAN connectomes (real MC only,
surrogates off). Used in §7d to check the direction of the MC–age relationship under
their preprocessing choice.

In [ ]:
if MIJALKOV_CACHE.exists() and not FORCE_RECOMPUTE:
    df_mij = load_dataframe(MIJALKOV_CACHE)
    print(f"Loaded Mijalkov cache ({len(df_mij)} subjects) → {MIJALKOV_CACHE}")
else:
    sub = df[df['dataset'] == MIJALKOV_DATASET].copy()
    if len(sub) == 0:
        print(f"No subjects with dataset=={MIJALKOV_DATASET}; skipping Mijalkov run.")
        df_mij = pd.DataFrame(columns=['sid','age','MC_Glob_thr'])
    else:
        sub = sub.reset_index(drop=True)
        sub['connectome'] = [proportional_threshold(c, MIJALKOV_DENSITY)
                             for c in sub['connectome'].values]
        compile_cpp(["esn_run"])
        export_connectomes(sub, "connectomes_mij.csv")
        write_config(in_csv="data/connectomes_mij.csv", out_csv="mc_mij.csv",
                     do_r="false", do_bs="false", do_uni="false", do_rsh="false",
                     n_rand_pairs=0)
        run_exe( ("esn_run.exe" if _is_windows() else "esn_run"))
        mij = pd.read_csv( "mc_mij.csv").rename(
            columns={"subject_id": "sid", "MC_Glob": "MC_Glob_thr", "MC_Bio": "MC_Bio_thr"})
        df_mij = sub[['sid','age']].merge(mij[['sid','MC_Glob_thr','MC_Bio_thr']],
                                          on="sid", how="left")
        df_mij.to_pickle(MIJALKOV_CACHE)
        print(f"Saved Mijalkov cache → {MIJALKOV_CACHE}")
df_mij.head()


## 5 · Figure 3 (primary) — *ESN performance and age as a function of disparity*

**Narrative (this drives the panel order).**

- **(a) The lifespan coupling.** Across development→aging, reservoir memory (`MC`) and
  structural **disparity** move together as a function of age; the inset shows the
  direct `MC ← disparity` dependence (with r²). *Establishes that performance and age
  are organised by disparity.*
- **(b) The signal is structural, not trivial.** Real `MC` is compared against weight
  **surrogates** (broken-stick / row-uniform / reshuffle). The gap between the real
  trajectory and the nulls shows that the connectome's weight organisation — not just
  its degree sequence or total strength — carries the computational effect.
- **(c) Which structure drives `MC`.** Among the theoretically linked quantities —
  **disparity, weighted clustering, communicability** (the low-order and full terms of
  the walk/communicability expansion) — we show their explanatory power for `MC`,
  **separately for development and aging**, exposing the regime change. *The full
  16-metric horse-race is deferred to the supplementary (§7f).*

Each panel is its own function drawing onto a supplied `ax`, so the layout/order is
trivial to rearrange in the assembly cell.

In [ ]:
# Headline disparity measure for Figure 3 (coherent with the PCA feature set).
DISP_COL   = FEATURES[0] if FEATURES else 'Ratio'           # 'Ratio'
DISP_LABEL = r'Disparity ($\Upsilon$)'
MAIN_PREDICTORS = [(DISP_COL, DISP_LABEL),
                   ('C_w',       r'Clustering ($C_w$)'),
                   ('comm_mean', r'Communicability $\langle G\rangle$')]
AGE_CMAP = 'plasma'                                          # firstpart decade colormap

def _sig(p):
    return '***' if p < 1e-3 else ('**' if p < 1e-2 else ('*' if p < 5e-2 else 'n.s.'))

# ── Panel (a): MC & disparity vs age (dual axis) + MC↔disparity inset ─────────
def panel_lifespan(ax):
    # left axis: MC_Glob lifespan trajectory (mean ± bootstrap CI)
    xa, ma, loa, hia = age_trajectory(df, 'MC_Glob')
    l1, = ax.plot(xa, ma, color='#1f4e79', lw=1.3, label='MC (global)')
    ax.fill_between(xa, loa, hia, color='#1f4e79', alpha=0.18, linewidth=0)
    ax.set_xlabel('Age (years)'); ax.set_ylabel('Memory capacity  MC', color='#1f4e79')
    ax.tick_params(axis='y', labelcolor='#1f4e79')

    # right axis: disparity lifespan trajectory
    ax2 = ax.twinx()
    xd, md_, lod, hid = age_trajectory(df, DISP_COL)
    l2, = ax2.plot(xd, md_, color='#c0392b', lw=1.3, ls='--', label=DISP_LABEL)
    ax2.fill_between(xd, lod, hid, color='#c0392b', alpha=0.15, linewidth=0)
    ax2.set_ylabel(DISP_LABEL, color='#c0392b'); ax2.tick_params(axis='y', labelcolor='#c0392b')

    # development / aging divider
    ax.axvline(AGE_SPLIT, color='0.4', lw=0.8, ls=':')
    ax.text(AGE_SPLIT, ax.get_ylim()[1], '  aging →', va='top', ha='left', fontsize=6, color='0.4')
    ax.text(AGE_SPLIT, ax.get_ylim()[1], '← devel.  ', va='top', ha='right', fontsize=6, color='0.4')
    ax.legend(handles=[l1, l2], loc='lower center', frameon=False, fontsize=6, ncol=2)

    # inset: direct MC ← disparity with OLS r²
    sub = df.dropna(subset=['MC_Glob', DISP_COL])
    axin = inset_axes(ax, width="38%", height="42%", loc='upper right', borderpad=0.6)
    sc = axin.scatter(sub[DISP_COL], sub['MC_Glob'], c=sub['age'], cmap=AGE_CMAP,
                      s=4, alpha=0.5, linewidths=0)
    b = np.polyfit(sub[DISP_COL], sub['MC_Glob'], 1)
    xs = np.linspace(sub[DISP_COL].min(), sub[DISP_COL].max(), 50)
    axin.plot(xs, np.polyval(b, xs), color='k', lw=1.0)
    r2, p = fit_linear(sub['MC_Glob'].values, sub[DISP_COL].values)
    axin.set_title(fr'MC $\leftarrow$ {DISP_LABEL}', fontsize=6, pad=2)
    axin.set_xlabel(DISP_LABEL, fontsize=5.5, labelpad=1)
    axin.set_ylabel('MC', fontsize=5.5, labelpad=1)
    axin.tick_params(labelsize=5, length=2)
    axin.text(0.04, 0.94, fr'$r^2$={r2:.2f} {_sig(p)}', transform=axin.transAxes,
              fontsize=5.5, va='top')
    sns.despine(ax=axin)
    sns.despine(ax=ax); sns.despine(ax=ax2, right=False)
    ax.set_title('(a)  Lifespan coupling of memory and disparity', loc='left', fontsize=7.5)

# ── Panel (b): real MC vs weight surrogates ──────────────────────────────────
def panel_surrogates(ax):
    null_specs = [('MC_Glob_BS',  'Broken-stick (disparity null)', '#27ae60'),
                  ('MC_Glob_Uni', 'Row-uniform (H0)',              '#8e44ad'),
                  ('MC_Glob_Rsh', 'Reshuffle (H1)',                '#e67e22')]
    have = [s for s in null_specs if s[0] in df.columns]
    if not have:
        ax.text(0.5, 0.5, 'Surrogates not computed\n(enable DO_BS/DO_UNI/DO_RSH in §0)',
                ha='center', va='center', fontsize=7, transform=ax.transAxes)
        ax.set_title('(b)  Memory vs weight surrogates', loc='left', fontsize=7.5)
        sns.despine(ax=ax); return
    xr, mr, lor, hir = age_trajectory(df, 'MC_Glob')
    ax.plot(xr, mr, color='#1f4e79', lw=1.6, label='Real', zorder=5)
    ax.fill_between(xr, lor, hir, color='#1f4e79', alpha=0.15, linewidth=0)
    for col, lab, c in have:
        x, m, lo, hi = age_trajectory(df, col)
        ax.plot(x, m, color=c, lw=1.0, alpha=0.9, label=lab)
        ax.fill_between(x, lo, hi, color=c, alpha=0.10, linewidth=0)
    ax.set_xlabel('Age (years)'); ax.set_ylabel('Memory capacity  MC')
    ax.axvline(AGE_SPLIT, color='0.4', lw=0.8, ls=':')
    ax.legend(loc='best', frameon=False, fontsize=5.5)
    ax.set_title('(b)  Memory is structural: real vs surrogates', loc='left', fontsize=7.5)
    sns.despine(ax=ax)

# ── Panel (c): structural predictors of MC, development vs aging ─────────────
def panel_predictors(ax):
    stages = [('Development', '#3498db'), ('Aging', '#e74c3c')]
    labels = [lab for _, lab in MAIN_PREDICTORS]
    xpos = np.arange(len(MAIN_PREDICTORS)); width = 0.38
    for j, (stage, col) in enumerate(stages):
        sub = df[df['stage'] == stage]
        r2s, sigs = [], []
        for metric, _ in MAIN_PREDICTORS:
            s = sub.dropna(subset=['MC_Glob', metric])
            if len(s) > 5:
                r2, p = fit_linear(s['MC_Glob'].values, s[metric].values)
            else:
                r2, p = np.nan, np.nan
            r2s.append(r2); sigs.append(_sig(p) if np.isfinite(p) else '')
        bars = ax.bar(xpos + (j - 0.5) * width, r2s, width, color=col, alpha=0.85, label=stage)
        for b, sg in zip(bars, sigs):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005, sg,
                    ha='center', va='bottom', fontsize=5.5)
    ax.set_xticks(xpos); ax.set_xticklabels(labels, fontsize=6.5)
    ax.set_ylabel(r'$R^2$  (MC $\sim$ metric)')
    ax.legend(frameon=False, fontsize=6, loc='upper right')
    ax.set_title('(c)  Structural drivers of memory', loc='left', fontsize=7.5)
    sns.despine(ax=ax)

print("Figure-3 panels defined.")


In [ ]:
# ── Assemble Figure 3 ────────────────────────────────────────────────────────
fig3 = plt.figure(figsize=(11, 7.5))
gs3  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.50)   # firstpart fig2 spacing
ax_a = fig3.add_subplot(gs3[0, :])     # (a) wide top
ax_b = fig3.add_subplot(gs3[1, 0])     # (b) bottom-left
ax_c = fig3.add_subplot(gs3[1, 1])     # (c) bottom-right

panel_lifespan(ax_a)
panel_surrogates(ax_b)
panel_predictors(ax_c)

fig3.savefig(FIG_DIR / 'figure3_performance_disparity.pdf', bbox_inches='tight', dpi=300)
fig3.savefig(FIG_DIR / 'figure3_performance_disparity.png', bbox_inches='tight', dpi=300)
print('saved figures/figure3_performance_disparity.{pdf,png}')
plt.show()


## 6 · Figure 4 (primary) — mediation, correlation structure, PCA & UMAP

**Narrative.** Having shown *that* disparity/clustering/communicability drive memory
(Fig 3c), Figure 4 asks *how the structure is organised*:

- **(a) Mediation placeholder.** Reserved for the mediation model
  *clustering + disparity → communicability* (your existing code drops in here).
- **(b) PCA biplot.** The feature set collapses onto a dominant axis aligned with age;
  loadings show which structural variables define it.
- **(c–d) Correlation structure, development vs aging.** Spearman correlations among the
  features differ between regimes — the same regime change seen in Fig 3c.
- **(e) UMAP coloured by age** — a continuous age gradient in the low-dimensional embedding.
- **(f) UMAP coloured by dataset** — *sanity check*: the embedding should reflect biology,
  not which cohort a subject came from.

Most of these are intended for the supplementary; the main text keeps (a), (b) and (f).

In [ ]:
FEAT_LABELS = {'Ratio': r'Disparity $\Upsilon$', 'MC_Glob': 'MC', 'MC_Glob_Real': 'MC',
               'comm_mean': r'Commun. $\langle G\rangle$', 'C_w': r'Clustering $C_w$',
               'age': 'Age', 'kY_obs': r'$k\Upsilon$'}

def _feature_matrix(sub):
    """Standardised feature matrix + the row index actually used (after dropna)."""
    X = sub[FEATURES].dropna()
    Xs = StandardScaler().fit_transform(X.values)
    return Xs, X.index

# ── Panel (a): mediation placeholder (left intentionally blank, per plan) ─────
def panel_mediation_placeholder(ax):
    ax.axis('off')
    ax.text(0.5, 0.5,
            'Mediation model\n(placeholder)\n\n'
            r'Clustering $C_w$ + Disparity $\Upsilon$' '\n'
            r'$\longrightarrow$ Communicability $\langle G\rangle$',
            ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.6', fc='#f4f4f4', ec='0.6', lw=0.8))
    ax.set_title('(a)  Structural mediation', loc='left', fontsize=7.5)

# ── Panel (b): pooled PCA biplot coloured by age ─────────────────────────────
def panel_pca_biplot(ax):
    Xs, idx = _feature_matrix(df)
    pca = PCA(n_components=len(FEATURES)).fit(Xs)
    Z = pca.transform(Xs)
    ages = df.loc[idx, 'age'].values
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=ages, cmap=AGE_CMAP, s=5, alpha=0.6, linewidths=0)
    # loading arrows (biplot)
    load = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
    scale = 0.9 * np.abs(Z[:, :2]).max() / (np.abs(load).max() + 1e-9)
    for i, feat in enumerate(FEATURES):
        ax.arrow(0, 0, load[i, 0]*scale, load[i, 1]*scale, color='k',
                 width=0.0, head_width=0.12, length_includes_head=True, alpha=0.8)
        ax.text(load[i, 0]*scale*1.12, load[i, 1]*scale*1.12,
                FEAT_LABELS.get(feat, feat), fontsize=5.5, ha='center', va='center')
    ax.set_xlim(np.array(ax.get_xlim()) * 1.18)   # headroom so labels clear the frame/title
    ax.set_ylim(np.array(ax.get_ylim()) * 1.18)
    ev = pca.explained_variance_ratio_ * 100
    ax.set_xlabel(f'PC1 ({ev[0]:.0f}%)'); ax.set_ylabel(f'PC2 ({ev[1]:.0f}%)')
    cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04); cb.set_label('Age', fontsize=6)
    cb.ax.tick_params(labelsize=5)
    ax.set_title('(b)  PCA biplot (coloured by age)', loc='left', fontsize=7.5)
    sns.despine(ax=ax)
    return pca

# ── Panels (c,d): Spearman correlation heatmaps, development vs aging ────────
def panel_corr(ax, stage, tag):
    sub = df[df['stage'] == stage][FEATURES].dropna()
    corr = sub.corr(method='spearman')
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    labs = [FEAT_LABELS.get(f, f) for f in FEATURES]
    ax.set_xticks(range(len(FEATURES))); ax.set_xticklabels(labs, rotation=45, ha='right', fontsize=5.5)
    ax.set_yticks(range(len(FEATURES))); ax.set_yticklabels(labs, fontsize=5.5)
    for i in range(len(FEATURES)):
        for j in range(len(FEATURES)):
            ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center',
                    fontsize=5, color='k' if abs(corr.values[i, j]) < 0.6 else 'w')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.ax.tick_params(labelsize=5)
    ax.set_title(f'({tag})  Correlations — {stage} (n={len(sub)})', loc='left', fontsize=7.5)

# ── Panels (e,f): UMAP coloured by age / dataset ─────────────────────────────
_UMAP_CACHE = {}
def _umap_embedding():
    """Compute (once) a pooled UMAP embedding on the standardised features."""
    if 'coords' in _UMAP_CACHE:
        return _UMAP_CACHE['coords'], _UMAP_CACHE['idx']
    Xs, idx = _feature_matrix(df)
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, n_components=2, random_state=SEED)
    coords = reducer.fit_transform(Xs)
    _UMAP_CACHE['coords'], _UMAP_CACHE['idx'] = coords, idx
    return coords, idx

def panel_umap(ax, color_by, tag):
    if not HAVE_UMAP:
        ax.text(0.5, 0.5, 'UMAP unavailable\n(`pip install umap-learn`)',
                ha='center', va='center', fontsize=7, transform=ax.transAxes)
        ax.set_title(f'({tag})  UMAP — coloured by {color_by}', loc='left', fontsize=7.5)
        ax.axis('off'); return
    coords, idx = _umap_embedding()
    if color_by == 'age':
        sc = ax.scatter(coords[:, 0], coords[:, 1], c=df.loc[idx, 'age'].values,
                        cmap=AGE_CMAP, s=5, alpha=0.6, linewidths=0)
        cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04); cb.set_label('Age', fontsize=6)
        cb.ax.tick_params(labelsize=5)
    else:  # dataset — the sanity check
        for d in sorted(df.loc[idx, 'dataset'].unique()):
            m = df.loc[idx, 'dataset'].values == d
            ax.scatter(coords[m, 0], coords[m, 1], s=5, alpha=0.6, linewidths=0,
                       color=ds2col[d], label=DS_NAMES.get(d, str(d)))
        ax.legend(frameon=False, fontsize=5, markerscale=1.5, loc='best', ncol=2)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title(f'({tag})  UMAP — coloured by {color_by}', loc='left', fontsize=7.5)
    sns.despine(ax=ax)

print("Figure-4 panels defined.")


In [ ]:
# ── Assemble Figure 4 ────────────────────────────────────────────────────────
fig4 = plt.figure(figsize=(11, 11))
gs4  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.50)
panel_mediation_placeholder(fig4.add_subplot(gs4[0, 0]))
pca = panel_pca_biplot(fig4.add_subplot(gs4[0, 1]))
panel_corr(fig4.add_subplot(gs4[1, 0]), 'Development', 'c')
panel_corr(fig4.add_subplot(gs4[1, 1]), 'Aging',       'd')
panel_umap(fig4.add_subplot(gs4[2, 0]), 'age',     'e')
panel_umap(fig4.add_subplot(gs4[2, 1]), 'dataset', 'f')

fig4.savefig(FIG_DIR / 'figure4_pca_umap.pdf', bbox_inches='tight', dpi=300)
fig4.savefig(FIG_DIR / 'figure4_pca_umap.png', bbox_inches='tight', dpi=300)
print('saved figures/figure4_pca_umap.{pdf,png}')
plt.show()


### 6b · Supplementary — PCA scree & loadings

Full PCA detail (scree + loadings heatmap + PC1–PC3) for the supplementary material.

In [ ]:
Xs_all, idx_all = _feature_matrix(df)
pca_full = PCA(n_components=len(FEATURES)).fit(Xs_all)
Z_all = pca_full.transform(Xs_all)
ev = pca_full.explained_variance_ratio_ * 100

figS, axS = plt.subplots(1, 3, figsize=(11, 3.2))
# scree
axS[0].bar(range(1, len(FEATURES)+1), ev, color=sns.color_palette(AGE_CMAP, len(FEATURES)))
axS[0].plot(range(1, len(FEATURES)+1), np.cumsum(ev), 'o-', color='0.2', lw=1.0)
axS[0].set_xticks(range(1, len(FEATURES)+1))
axS[0].set_xlabel('Principal component'); axS[0].set_ylabel('Variance explained (%)')
axS[0].set_title('Scree', loc='left', fontsize=7.5); sns.despine(ax=axS[0])
# loadings heatmap
im = axS[1].imshow(pca_full.components_, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axS[1].set_yticks(range(len(FEATURES)))
axS[1].set_yticklabels([f'PC{i+1}\n({ev[i]:.0f}%)' for i in range(len(FEATURES))], fontsize=5.5)
axS[1].set_xticks(range(len(FEATURES)))
axS[1].set_xticklabels([FEAT_LABELS.get(f, f) for f in FEATURES], rotation=45, ha='right', fontsize=5.5)
plt.colorbar(im, ax=axS[1], fraction=0.046, pad=0.04).ax.tick_params(labelsize=5)
axS[1].set_title('Loadings', loc='left', fontsize=7.5)
# PC1 vs PC3
if len(FEATURES) >= 3:
    sc = axS[2].scatter(Z_all[:, 0], Z_all[:, 2], c=df.loc[idx_all, 'age'].values,
                        cmap=AGE_CMAP, s=5, alpha=0.6, linewidths=0)
    axS[2].set_xlabel(f'PC1 ({ev[0]:.0f}%)'); axS[2].set_ylabel(f'PC3 ({ev[2]:.0f}%)')
    plt.colorbar(sc, ax=axS[2], fraction=0.046, pad=0.04).set_label('Age', fontsize=6)
axS[2].set_title('PC1 vs PC3', loc='left', fontsize=7.5); sns.despine(ax=axS[2])
figS.tight_layout()
figS.savefig(FIG_DIR / 'figureS_pca_detail.pdf', bbox_inches='tight', dpi=300)
print('saved figures/figureS_pca_detail.pdf')
plt.show()


## 7 · Secondary analyses (computed here; one or two sentences each in the text)

These support the figures but live mostly in the supplementary: the null-model table,
the biological random-pair null, the IPC order decomposition, the Mijalkov direction
check, the r-vs-r² consistency check, and (optional) the full 16-metric horse-race.

### 7a · MC null-model summary (3 surrogates)

Ratios `MC_real / MC_null` per stage. A ratio above 1 means the real weight structure
*helps* memory relative to that null; the deviation pattern across nulls localises the
effect (heterogeneity vs wiring).

In [ ]:
ratio_cols = {'Ratio_Glob_BS':'Broken-stick (disparity)',
              'Ratio_Glob_Uni':'Row-uniform (H0)',
              'Ratio_Glob_Rsh':'Reshuffle (H1)'}
have_ratio = {k: v for k, v in ratio_cols.items() if k in df.columns}
if have_ratio:
    rows = []
    for stage in ['Development', 'Aging']:
        sub = df[df['stage'] == stage]
        for col, lab in have_ratio.items():
            v = sub[col].dropna()
            t, p = stats.ttest_1samp(v, 1.0) if len(v) > 2 else (np.nan, np.nan)
            rows.append(dict(Stage=stage, Null=lab, n=len(v),
                             mean_ratio=round(v.mean(), 3), std=round(v.std(), 3),
                             p_vs_1=f'{p:.1e}' if np.isfinite(p) else 'n/a'))
    null_table = pd.DataFrame(rows)
    null_table.to_csv(FIG_DIR / 'table_null_models.csv', index=False)
    print('saved figures/table_null_models.csv')
    display(null_table)
else:
    print('Surrogate ratio columns not present — enable DO_BS/DO_UNI/DO_RSH in §0 and recompute.')


### 7b · Biological random-pair null

For the thalamic (bio) input we compare `MC_Bio` against `N_RAND_PAIRS` random input
pairs (`MC_Rand_*`). A positive `Z_Score` indicates the thalamic projection yields more
memory than typical random node pairs — i.e. the result is specific to the biological
input site, **not** an artefact of projecting to two nodes (this is distinct from the
thalamic pair 76–77 used as the input itself).

In [ ]:
if 'Z_Score' in df.columns and df['Z_Score'].notna().any():
    fig, axz = plt.subplots(1, 2, figsize=(7.5, 3.0))
    for stage, c in [('Development', '#3498db'), ('Aging', '#e74c3c')]:
        z = df.loc[df['stage'] == stage, 'Z_Score'].dropna()
        axz[0].hist(z, bins=30, alpha=0.55, color=c, label=f'{stage} (n={len(z)})', density=True)
    axz[0].axvline(0, color='0.3', lw=0.8, ls=':')
    axz[0].set_xlabel('Z (MC$_{bio}$ vs random pairs)'); axz[0].set_ylabel('density')
    axz[0].legend(frameon=False, fontsize=6); sns.despine(ax=axz[0])
    axz[0].set_title('Random-pair null: Z distribution', loc='left', fontsize=7.5)

    xz, mz, loz, hiz = age_trajectory(df, 'Z_Score')
    axz[1].plot(xz, mz, color='#16a085', lw=1.2)
    axz[1].fill_between(xz, loz, hiz, color='#16a085', alpha=0.15, linewidth=0)
    axz[1].axhline(0, color='0.3', lw=0.8, ls=':')
    axz[1].set_xlabel('Age (years)'); axz[1].set_ylabel('Z')
    axz[1].set_title('Z across the lifespan', loc='left', fontsize=7.5); sns.despine(ax=axz[1])
    fig.tight_layout(); fig.savefig(FIG_DIR / 'figureS_randompair_null.pdf', bbox_inches='tight', dpi=300)
    print(f"Mean Z = {df['Z_Score'].mean():.2f} (fraction Z>0: {(df['Z_Score']>0).mean():.0%})")
    plt.show()
else:
    print('Random-pair null not computed — set N_RAND_PAIRS>0 in §0 and recompute §4b.')


### 7c · IPC order decomposition (small τ)

Linear (P1) vs quadratic (P2) vs cross capacity. The point is that the **linear term
dominates** (so MC is the right headline metric) and that the decomposition is
**consistent** across development and aging.

In [ ]:
ipc_terms = [('Lin_Glob', 'Linear (P1)'), ('Quad_Glob', 'Quadratic (P2)'),
             ('Cross11_Glob', 'Cross P1·P1')]
ipc_terms = [(c, l) for c, l in ipc_terms if c in df_ipc.columns]
if ipc_terms:
    summ = (df_ipc.groupby('stage')[[c for c, _ in ipc_terms]].mean()
            .reindex(['Development', 'Aging']))
    print('Mean IPC capacity by stage (global input, τ =', IPC_TAU, '):')
    display(summ.round(3))
    fig, axi = plt.subplots(figsize=(5.0, 3.0))
    xpos = np.arange(len(ipc_terms)); width = 0.38
    for j, stage in enumerate(['Development', 'Aging']):
        if stage in summ.index:
            vals = summ.loc[stage].values
            axi.bar(xpos + (j - 0.5) * width, vals, width,
                    label=stage, color=['#3498db', '#e74c3c'][j], alpha=0.85)
    axi.set_xticks(xpos); axi.set_xticklabels([l for _, l in ipc_terms], fontsize=6.5)
    axi.set_ylabel('Capacity (Σ r²)'); axi.legend(frameon=False, fontsize=6)
    axi.set_title('IPC order decomposition', loc='left', fontsize=7.5); sns.despine(ax=axi)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'figureS_ipc_orders.pdf', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('IPC columns not present — run §4c.')


### 7d · Mijalkov et al. direction check

Mijalkov et al. use **FA-weighted** connectomes; here we use **streamline counts with a
proportional threshold**. On the thresholded camCAN data we check the **sign** of the
MC–age relationship to state, in one sentence, whether our preprocessing reproduces a
positive MC–age association.

In [ ]:
if len(df_mij) and df_mij['MC_Glob_thr'].notna().any():
    s = df_mij.dropna(subset=['MC_Glob_thr', 'age'])
    r, p = pearsonr(s['age'], s['MC_Glob_thr'])
    direction = 'POSITIVE' if r > 0 else 'negative'
    print(f"camCAN, proportional threshold density={MIJALKOV_DENSITY}, n={len(s)}")
    print(f"  corr(age, MC_thresholded) : r={r:.3f}  (r²={r**2:.3f}), p={p:.1e}  → {direction}")
    fig, axm = plt.subplots(figsize=(4.2, 3.2))
    axm.scatter(s['age'], s['MC_Glob_thr'], s=6, alpha=0.5, color='#8e44ad', linewidths=0)
    b = np.polyfit(s['age'], s['MC_Glob_thr'], 1)
    xs = np.linspace(s['age'].min(), s['age'].max(), 50)
    axm.plot(xs, np.polyval(b, xs), 'k-', lw=1.0)
    axm.set_xlabel('Age (years)'); axm.set_ylabel('MC (thresholded)')
    axm.text(0.04, 0.95, fr'$r$={r:.2f}, $r^2$={r**2:.2f}', transform=axm.transAxes,
             va='top', fontsize=6)
    axm.set_title('Mijalkov replication (camCAN)', loc='left', fontsize=7.5); sns.despine(ax=axm)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'figureS_mijalkov.pdf', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No Mijalkov result — check §4d (dataset id / availability).')


### 7e · r vs r² consistency

We standardise on r² (Damicelli). This confirms the Σ|r| variant (Suárez) is a
monotone re-scaling, so the choice does not change any qualitative conclusion.

In [ ]:
if 'MC_Glob_r' in df.columns and df['MC_Glob_r'].notna().any():
    s = df.dropna(subset=['MC_Glob', 'MC_Glob_r'])
    rr, pp = pearsonr(s['MC_Glob'], s['MC_Glob_r'])
    rho, _ = spearmanr(s['MC_Glob'], s['MC_Glob_r'])
    fig, axr = plt.subplots(figsize=(3.6, 3.4))
    axr.scatter(s['MC_Glob_r'], s['MC_Glob'], s=6, alpha=0.5, color='#2c3e50', linewidths=0)
    axr.set_xlabel(r'MC ($\Sigma|r|$, Suárez)'); axr.set_ylabel(r'MC ($\Sigma r^2$, Damicelli)')
    axr.text(0.04, 0.95, f'Pearson r={rr:.3f}\nSpearman ρ={rho:.3f}',
             transform=axr.transAxes, va='top', fontsize=6)
    axr.set_title('r vs r² agreement', loc='left', fontsize=7.5); sns.despine(ax=axr)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'figureS_r_vs_r2.pdf', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('Σ|r| variant not computed — set DO_R=True in §0 and recompute §4b.')


### 7f · Full structural horse-race (supplementary, optional)

The complete `R²(MC ~ metric)` ranking over ~15 graph metrics, development vs aging.
Off by default (`DO_FULL_HORSERACE=False`) because it runs `networkx` on every
connectome. **Navigability** is included only if your `helperfuncs.py`
(`weight_greedy_nav`) is importable, to avoid second-guessing your routing definition.

In [ ]:
def graph_metrics(A, rho=SPECTRAL_RADIUS, bc_k=50):
    A = np.asarray(A, float).copy(); np.fill_diagonal(A, 0)
    n = A.shape[0]
    deg = (A > 0).sum(1).astype(float); pos = A[A > 0]
    out = dict(
        density=float((A > 0).sum() / (n * (n - 1))),
        mean_weight=float(pos.mean()) if pos.size else 0.0,
        weight_var=float(pos.var()) if pos.size else 0.0,
        mean_degree=float(deg.mean()), degree_var=float(deg.var()),
        spectral_radius=float(np.abs(eigvals(A)).max()),
        clustering=onnela_clustering_global(A),
        communicability=communicability_mean(A, rho),
    )
    if HAVE_NX:
        Gu = nx.from_numpy_array((A > 0).astype(float))
        Gd = nx.from_numpy_array(A)
        for _, _, d in Gd.edges(data=True):
            d['distance'] = 1.0 / d['weight'] if d['weight'] > 0 else np.inf
        try: out['global_efficiency'] = nx.global_efficiency(Gu)
        except Exception: out['global_efficiency'] = np.nan
        try: out['transitivity'] = nx.transitivity(Gu)
        except Exception: out['transitivity'] = np.nan
        try: out['assortativity'] = nx.degree_assortativity_coefficient(Gu)
        except Exception: out['assortativity'] = np.nan
        try:
            comm = nx.community.greedy_modularity_communities(Gu)
            out['modularity'] = nx.community.modularity(Gu, comm)
        except Exception: out['modularity'] = np.nan
        try:
            bc = nx.betweenness_centrality(Gd, weight='distance', k=min(bc_k, n), seed=SEED)
            out['betweenness'] = float(np.mean(list(bc.values())))
        except Exception: out['betweenness'] = np.nan
        try:
            rc = nx.rich_club_coefficient(Gu, normalized=False)
            out['rich_club'] = float(np.mean(list(rc.values()))) if rc else np.nan
        except Exception: out['rich_club'] = np.nan
    try:
        from helperfuncs import weight_greedy_nav
        out['navigability'] = float(weight_greedy_nav(A))
    except Exception:
        pass
    return out

if DO_FULL_HORSERACE:
    if HORSERACE_CACHE.exists() and not FORCE_RECOMPUTE:
        dfm = load_dataframe(HORSERACE_CACHE)
        print('loaded horse-race cache')
    else:
        from tqdm import tqdm
        recs = [graph_metrics(np.asarray(c, float)) for c in tqdm(df['connectome'].values, ncols=70)]
        dfm = pd.DataFrame(recs); dfm['sid'] = df['sid'].values
        dfm.to_pickle(HORSERACE_CACHE); print('cached horse-race metrics')
    metric_cols = [c for c in dfm.columns if c != 'sid']
    work = df[['sid', 'MC_Glob', 'stage']].merge(dfm, on='sid')
    LAB = {'density':'Density','mean_weight':'Mean weight','weight_var':'Weight var',
           'spectral_radius':'Spectral radius','mean_degree':'Mean degree','degree_var':'Degree var',
           'global_efficiency':'Global efficiency','clustering':'Clustering','modularity':'Modularity',
           'assortativity':'Assortativity','betweenness':'Betweenness','transitivity':'Transitivity',
           'rich_club':'Rich club','communicability':'Communicability','navigability':'Navigability'}
    order = (work[work.stage=='Aging'][['MC_Glob']+metric_cols].dropna()
             .corr()['MC_Glob'].pow(2).drop('MC_Glob').sort_values())
    metric_cols = list(order.index)
    fig, axh = plt.subplots(figsize=(6.5, 0.32*len(metric_cols)+1))
    y = np.arange(len(metric_cols)); h = 0.38
    for j, stage in enumerate(['Development', 'Aging']):
        sub = work[work.stage == stage]
        r2 = [fit_linear(sub.dropna(subset=['MC_Glob', m])['MC_Glob'].values,
                         sub.dropna(subset=['MC_Glob', m])[m].values)[0]
              if sub[m].notna().sum() > 5 else np.nan for m in metric_cols]
        axh.barh(y + (j-0.5)*h, r2, h, color=['#3498db', '#e74c3c'][j], alpha=0.85, label=stage)
    axh.set_yticks(y); axh.set_yticklabels([LAB.get(m, m) for m in metric_cols], fontsize=6)
    axh.set_xlabel(r'$R^2$ (MC $\sim$ metric)'); axh.legend(frameon=False, fontsize=6)
    axh.set_title('Full structural horse-race', loc='left', fontsize=7.5); sns.despine(ax=axh)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'figureS_horserace.pdf', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('DO_FULL_HORSERACE=False — set it True in §0 to compute the supplementary ranking.')


## 8 · Outputs, decisions to confirm, and how to scale the compute

**Saved to `figures/`:** `figure3_performance_disparity.{pdf,png}`,
`figure4_pca_umap.{pdf,png}`, and supplementary `figureS_*.pdf` +
`table_null_models.csv`. Heavy results are cached in `cache/*.pkl`.

**Decisions worth confirming (each is a one-line change in §0):**

- **`AGE_SPLIT = 33`** matches both prior notebooks; the manuscript text suggests a
  turning point nearer 30–32. Re-run with 30 to check robustness of Fig 3c / Fig 4c–d.
- **Binning.** Lifespan curves use `groupby('age').mean()` with bootstrap CIs and
  continuous scatters with OLS r² — **no median/quantile bins**, per `firstpart.ipynb`.
- **Disparity column.** Figure 3 uses `DISP_COL = FEATURES[0]` (`'Ratio'`) so it is
  coherent with the PCA feature set; switch to `'kY_obs'` if you prefer the raw k·Υ.
- **Reshuffle null** defaults to node-level (paper H1); set `RESHUFFLE_MODE='global'`
  for the whole-network variant.
- **Mediation panel (Fig 4a)** is a deliberate placeholder for your existing model.

**Scaling the compute (the MC sweep in §4b is the only slow part):**

1. First pass — real MC only: set `DO_BS=DO_UNI=DO_RSH=False`, `N_RAND_PAIRS=0`. Fast;
   gives Fig 3a and the MC columns.
2. Add the nulls (Fig 3b, §7a/7b) by switching those back on and re-running §4b — the
   cache makes it the only cell that recomputes.
3. Use `SUBSET_N` for an end-to-end smoke test on a handful of subjects before the full
   run, and `N_WORKERS`/OpenMP threads to use all cores.

**Reproducibility.** All randomness (W_in, surrogates, UMAP) is seeded from `SEED`;
the C++ seeds each ESN per subject, so a given `config.txt` reproduces bit-for-bit.
